In [1]:
import pandas as pd
import numpy as np
import time
from concurrent.futures import ThreadPoolExecutor

FILE_PATH = r'E:\phivolcs_earthquake_data.csv'
df_raw = pd.read_csv(FILE_PATH)

df = pd.concat([df_raw] * 10, ignore_index=True)

In [2]:
def process_data(data):
    # Cleans up data with errors='coerce'
    data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')
    data['Depth_In_Km'] = pd.to_numeric(data['Depth_In_Km'], errors='coerce')
    
    # Filtering for Earthquakes with a Magnitude of >= 4
    filtered = data[data['Magnitude'] >= 4.0].copy()
    
    # Aggregation - Grouping by Location
    agg = filtered.groupby('General_Location').agg(
        count=('Magnitude', 'count'),
        sum_mag=('Magnitude', 'sum'),
        max_depth=('Depth_In_Km', 'max')
    ).reset_index()
    
    return agg

In [3]:
# Finalizing of global aggregation and sorting
def finalize_and_sort(agg_df):
    final = agg_df.groupby('General_Location').agg({
        'count': 'sum',
        'sum_mag': 'sum',
        'max_depth': 'max'
    }).reset_index()
    
    final['mean_magnitude'] = final['sum_mag'] / final['count']
    
    # Sorting, going descending order
    final_sorted = final.sort_values(by=['count', 'mean_magnitude'], ascending=False).reset_index(drop=True)
    return final_sorted[['General_Location', 'count', 'mean_magnitude', 'max_depth']]

In [4]:
# SEQUENTIAL VERSION
start_seq = time.time()
seq_intermediate = process_data(df)
seq_results = finalize_and_sort(seq_intermediate)
seq_time = time.time() - start_seq

In [5]:
# MULTITHREADING VERSION
n_workers = 4
start_par = time.time()
chunks = np.array_split(df, n_workers)

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    chunk_results = list(executor.map(process_data, chunks))

combined_chunks = pd.concat(chunk_results)
par_results = finalize_and_sort(combined_chunks)
par_time = time.time() - start_par

G:\E\anaconda\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [6]:
# Performance comparison
performance_df = pd.DataFrame({
    "Version": ["Sequential", f"Parallel ({n_workers} Threads)"],
    "Execution Time (s)": [seq_time, par_time],
    "Speedup": [1.0, seq_time / par_time]
})

# Displays comparison
print("--- SEQUENTIAL RESULTS ---")
print(seq_results.head(5))
print("\n--- PARALLEL RESULTS ---")
print(par_results.head(5))
print("\n--- PERFORMANCE COMPARISON ---")
print(performance_df)

--- SEQUENTIAL RESULTS ---
    General_Location  count  mean_magnitude  max_depth
0   Davao Occidental  12970        4.468620      579.0
1    Surigao Del Sur   6630        4.469231      134.0
2     Davao Oriental   5440        4.508824      181.0
3  Surigao Del Norte   2840        4.395423      131.0
4      Eastern Samar   1870        4.380749      103.0

--- PARALLEL RESULTS ---
    General_Location  count  mean_magnitude  max_depth
0   Davao Occidental  12970        4.468620      579.0
1    Surigao Del Sur   6630        4.469231      134.0
2     Davao Oriental   5440        4.508824      181.0
3  Surigao Del Norte   2840        4.395423      131.0
4      Eastern Samar   1870        4.380749      103.0

--- PERFORMANCE COMPARISON ---
                Version  Execution Time (s)   Speedup
0            Sequential            0.262076  1.000000
1  Parallel (4 Threads)            0.157915  1.659605
